# 02 — Dữ liệu thị trường, vĩ mô và tài chính

Dự báo log return phiên kế tiếp, sau đó suy ra giá và hướng. Chạy các ô từ trên xuống. Dữ liệu dừng tại 30/09/2025.

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Lấy từ đâu và dùng vào việc gì?

| Nhóm | Nguồn/cách lấy | Ý nghĩa khi thử đưa vào mô hình |
|---|---|---|
| VN-Index | API biểu đồ DNSE, lưu OHLCV chỉ số | Log return thị trường và độ biến động 20 phiên |
| USD/VND, XAU/USD | CSV lịch sử Investing.com | Tỷ giá và giá vàng quốc tế quy đổi VND/ounce |
| CPI | Bản lưu lịch công bố Investing.com, cột Thực tế | Lạm phát so cùng kỳ, đơn vị phần trăm |
| GDP | World Bank, chỉ tiêu `NY.GDP.MKTP.CD` | Quy mô GDP danh nghĩa năm theo USD |
| Báo cáo quý FPT | Bản xuất báo cáo VCI qua vnstock | Doanh thu thuần, lợi nhuận gộp, EBIT, lãi thuộc cổ đông mẹ |
| Cổ phiếu lưu hành | Báo cáo/công bố chính thức FPT, trích mốc kèm URL | Số cổ phiếu lưu hành theo thông tin đã ghi nhận |

`--from-cache` là chuẩn hóa và ghép **bản lưu hiện có**, không phải tự động tải mọi website. Các bước xuất CSV và trường bắt buộc nằm trong [sổ tay dữ liệu](../data/README.md). Việc thêm biến nhằm kiểm tra có thông tin hữu ích hay không; không mặc định cứ nhiều biến là tốt hơn.

## Ghép từ ngày có thể biết

- Biến ngày được dùng từ ngày lịch sau; giá trị cũ quá 10 ngày bị bỏ.
- CPI dùng ngày công bố + 1, giữ tối đa 65 ngày từ hiệu lực.
- Báo cáo quý dùng cuối quý + 45 ngày, giữ tối đa 150 ngày; đây là giả định hiệu lực.
- GDP dùng cuối năm + 180 ngày, giữ tối đa 550 ngày; không biết trước GDP ngay đầu năm.
- Cổ phiếu lưu hành dùng ngày tham chiếu/công bố riêng + 1, giữ tối đa 550 ngày.

Ví dụ quý II/2025 kết thúc 30/06, cộng 45 ngày là 14/08/2025: trước 14/08 không đưa giá trị quý II vào hàng đầu vào. **Ghép lùi** nghĩa là mỗi ngày chỉ tìm giá trị đã có hiệu lực gần nhất; tuyệt đối không lấy mốc công bố sau đó để lấp ngược.

In [ ]:
from fetch_supplemental import run_from_cache, RAW_DIR, DEFAULT_OUTPUT, MODEL_SUPPLEMENTAL_COLUMNS
# Chuẩn hóa các bản nguồn đã lưu; bảng hiển thị tập trung vào đầu vào mô hình.
audit = run_from_cache(RAW_DIR, DEFAULT_OUTPUT)
print("Số phiên FPT:", audit["rows"], "| Mốc cuối:", audit["cutoff_date"])
supp = pd.read_csv(DEFAULT_OUTPUT)[["date", *MODEL_SUPPLEMENTAL_COLUMNS]]
display(supp.tail())


## Độ phủ: không phải nguồn nào cũng có từ 2007

Vàng bắt đầu 01/01/2019; CPI có ngày công bố đầu 27/12/2018; GDP chỉ có 15 năm 2010–2024; báo cáo quý có 52 quan sát nhưng thiếu quý III/2007–quý IV/2012. Cổ phiếu lưu hành chỉ có 10 mốc, không phải sổ đăng ký cập nhật hàng ngày.

Số ô thiếu được đếm **sau ghép lên 4.671 ngày FPT**: vàng 2.988, CPI 2.987, GDP 1.110, mỗi chỉ tiêu quý 1.407, cổ phiếu lưu hành 2.859. VN-Index có 5 phiên thiếu nội bộ ở 01–05/02/2021, ngoài số hàng đầu chưa tính được return/volatility. Ngày đầu và cuối đúng không có nghĩa ở giữa không đứt đoạn.

Ô dưới chỉ hiện các biến thực sự được đưa vào mô hình. Thiếu trước khi có nguồn và thiếu do số liệu đã quá cũ được giữ là thiếu. Bước `prepare` mới chọn mẫu chung có đủ 60 phiên liên tiếp; không nội suy một đường nối đến số liệu công bố trong tương lai.

Đọc tiếp [EDA và công thức](../docs/EDA_VA_DAC_TRUNG.md), [cẩm nang dùng chung](../docs/CAM_NANG_HIEU_DU_AN.md).

In [ ]:
display(supp.isna().sum().to_frame('Số ô thiếu'))

## Rà soát phần thiếu mà bảng cuối có thể che đi

Ô trống chỉ là một dạng thiếu: nguồn có thể mất cả dòng. Bản VN-Index hiện thiếu 36 ngày có trong FPT; giữ giá trị cũ sau ghép chỉ còn 5 ô thiếu nội bộ. Return giữa hai hàng nguồn đi qua ngày bị khuyết có thể là nhiều phiên. Báo cáo quý cũng thiếu 22 kỳ dù 52 hàng đang có không thiếu số.

Ô dưới **chỉ đọc** các tệp và hiện kết quả, không tự sửa hoặc điền giá trị. Danh sách nguồn để tìm thêm và bước nghiệm thu file mới ở [hướng dẫn kiểm tra dữ liệu](../docs/HUONG_DAN_KIEM_TRA_VA_BO_SUNG_DU_LIEU.md).


In [ ]:
from audit_data_quality import review
quality = review(ROOT)
print("Ngày FPT chưa có trong nguồn VN-Index:", len(quality["calendar_comparison"]["fpt_dates_absent_from_vnindex"]))
print("Quý tài chính còn thiếu:", quality["financial_missing_quarters_inside_archive"])
coverage = pd.DataFrame(quality["aligned_model_feature_coverage"]).T
display(coverage[["first_valid_session", "last_valid_session", "missing_rows", "internal_missing_rows"]])
